# Audit a colleague's cleaned data

*Data Analysis in Python · Topic 3.5*

A colleague sent their "cleaned" sessions table for January to June 2024, and it doesn't match the analytics platform's report of 464,344 sessions. Find out why, with checks that say what clean means.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell and the two cells that build `clean` (the lesson's steps) and `colleague` (your colleague's table).
2. In the first cell marked ✏️, write a function `validate(df, start, end)` that returns a dictionary with these six keys, each `True` or `False`:
   - `"dates in range"`: every date is from `start` to `end`, both included
   - `"no duplicate rows"`: no row repeats another
   - `"no test traffic"`: no row's `session_medium` is `test`
   - `"no missing campaigns"`: no `session_campaign` is missing
   - `"numbers are numbers"`: `sessions`, `transactions` and `revenue` are numeric columns
   - `"transactions within sessions"`: no row has more transactions than sessions
3. In the second cell marked ✏️, create `clean_checks` and `colleague_checks` with `validate()`, using `START` and `END`. Add an `assert` that stops the notebook if any of `clean_checks` fails, with the checks as its message.
4. In the third cell marked ✏️, create:
   - `reconciled`: whether `clean`'s total sessions equal `REPORTED_SESSIONS`
   - `colleague_gap`: how many more sessions `colleague` has than `REPORTED_SESSIONS`
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output. It also runs your `validate()` on copies of `clean` that it breaks on purpose, one rule at a time.

**Expected output**: `validate(clean, START, END)` returns six checks, all `True`. In `colleague_checks`, `dates in range`, `no duplicate rows`, `no test traffic` and `no missing campaigns` are `False`, and `numbers are numbers` and `transactions within sessions` are `True`. `reconciled` is `True` and `colleague_gap` is `30590`.

## Parameters

In [ ]:
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = ["2024-01", "2024-02", "2024-03", "2024-04", "2024-05", "2024-06"]
START, END = "2024-01-01", "2024-06-30"
REPORTED_SESSIONS = 464344  # the analytics platform's report for January to June 2024

## The two tables

In [ ]:
def read_month(month):
    return pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv", parse_dates=["date"],
                       na_values={"session_campaign": ["(not set)", "-"]})


clean = pd.concat([read_month(month) for month in MONTHS], ignore_index=True)
clean["session_campaign"] = clean["session_campaign"].fillna("(not set)")
clean = clean.drop_duplicates()
clean = clean[(clean["date"] <= END) & (clean["session_medium"] != "test")]
len(clean)

In [ ]:
# Your colleague's table: the six files, stacked
colleague = pd.concat([pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv", parse_dates=["date"])
                       for month in MONTHS], ignore_index=True)
len(colleague)

## Validate

In [ ]:
# ✏️ Write validate(df, start, end)


In [ ]:
# ✏️ Create clean_checks and colleague_checks, and assert that clean passes


## Reconcile

In [ ]:
# ✏️ Create reconciled and colleague_gap


In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything: it breaks copies of clean.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    names = ["dates in range", "no duplicate rows", "no test traffic", "no missing campaigns", "numbers are numbers", "transactions within sessions"]
    validate = found.get("validate")
    clean = found.get("clean")
    start, end = found.get("START"), found.get("END")

    def run(df):
        checks = validate(df, start, end)
        if not isinstance(checks, dict) or sorted(checks) != sorted(names):
            raise ValueError(f"validate() should return a dictionary with the six keys from the brief. It returned the keys {sorted(checks) if isinstance(checks, dict) else checks!r}.")
        return {name: bool(checks[name]) for name in names}

    works = False
    if not callable(validate):
        results.append((False, "validate isn't a function yet. Define it with def validate(df, start, end): and return the dictionary."))
    elif not isinstance(clean, pd.DataFrame) or len(clean) != 10181:
        results.append((False, "clean should have 10,181 rows. Run the cell that builds it, and don't change it in your cells."))
    else:
        try:
            on_clean = run(clean)
        except Exception as error:
            hint = " A single date can't be compared with text such as START: use df[\"date\"].between(start, end).all()." if "Timestamp" in str(error) else ""
            results.append((False, f"validate(clean, START, END) stopped with {type(error).__name__}: {error}.{hint}"))
        else:
            if all(on_clean.values()):
                works = True
                results.append((True, "validate() passes the cleaned table on all six checks."))
            else:
                failing = [name for name in names if not on_clean[name]]
                results.append((False, f"validate() fails the cleaned table on {failing}, but clean meets those rules. Check that the condition says what the rule says (use between() for the dates: comparing a date with text works on a column, not on a single date)."))

    if works:
        first = clean.index[0]

        def broken(name):
            df = clean.copy()
            if name == "dates in range":
                df.loc[first, "date"] = pd.Timestamp(end) + pd.Timedelta(days=1)
            elif name == "no duplicate rows":
                df = pd.concat([df, df.iloc[[0]]])
            elif name == "no test traffic":
                df.loc[first, "session_medium"] = "test"
            elif name == "no missing campaigns":
                df.loc[first, "session_campaign"] = None
            elif name == "numbers are numbers":
                df["revenue"] = df["revenue"].map(lambda value: f"${value:,.2f}")
            elif name == "transactions within sessions":
                df.loc[first, "transactions"] = df.loc[first, "sessions"] + 1
            return df

        missed = []
        for name in names:
            try:
                got = run(broken(name))
            except Exception as error:
                missed.append(f"{name} (validate() stopped with {type(error).__name__})")
                continue
            if got[name] or not all(got[other] for other in names if other != name):
                missed.append(name)
        if missed:
            results.append((False, f"Your checks don't catch a table broken on purpose, one rule at a time: {missed}. Each check should be False when its rule is broken, and only then."))
        else:
            results.append((True, "Every check fails when its rule is broken, and only then: the checks work."))

    clean_checks = found.get("clean_checks")
    colleague_checks = found.get("colleague_checks")
    expected = {"dates in range": False, "no duplicate rows": False, "no test traffic": False, "no missing campaigns": False, "numbers are numbers": True, "transactions within sessions": True}
    if not isinstance(clean_checks, dict) or not all(bool(value) for value in clean_checks.values()):
        results.append((False, "clean_checks should be validate(clean, START, END), with every check True."))
    elif not isinstance(colleague_checks, dict) or sorted(colleague_checks) != sorted(names):
        results.append((False, "colleague_checks should be validate(colleague, START, END)."))
    elif {name: bool(colleague_checks[name]) for name in names} == expected:
        results.append((True, "The colleague's table fails four checks: dates outside June (July 1 and 2), duplicates, test traffic and missing campaigns."))
    else:
        results.append((False, f"colleague_checks should fail dates, duplicates, test traffic and missing campaigns. It is { {name: bool(colleague_checks[name]) for name in names} }."))
    if history is not None and not any("assert " in source for source in history):
        results.append((False, "Add an assert after clean_checks: assert all(clean_checks.values()), clean_checks."))

    reconciled = found.get("reconciled")
    gap = found.get("colleague_gap")
    if pd.api.types.is_bool(reconciled) and bool(reconciled):
        results.append((True, "clean's 464,344 sessions match the platform's report."))
    else:
        results.append((False, f"reconciled should be clean[\"sessions\"].sum() == REPORTED_SESSIONS, which is True. It is {reconciled!r}."))
    if isinstance(gap, (pd.DataFrame, pd.Series)):
        results.append((False, f"colleague_gap is a {type(gap).__name__}, not a number. Subtract two totals: colleague[\"sessions\"].sum() - REPORTED_SESSIONS."))
    elif gap == 30590:
        results.append((True, "The colleague's table has 30,590 sessions too many: the copies, the July days and the test traffic."))
    elif gap == -30590:
        results.append((False, "colleague_gap is negative. Subtract REPORTED_SESSIONS from the colleague's total, not the other way round."))
    else:
        results.append((False, f"colleague_gap should be colleague[\"sessions\"].sum() - REPORTED_SESSIONS. It is {gap!r}."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
def validate(df, start, end):
    """Return each check's name and whether the table passes it."""
    return {
        "dates in range": bool(df["date"].between(start, end).all()),
        "no duplicate rows": not df.duplicated().any(),
        "no test traffic": not (df["session_medium"] == "test").any(),
        "no missing campaigns": not df["session_campaign"].isna().any(),
        "numbers are numbers": all(pd.api.types.is_numeric_dtype(df[column]) for column in ["sessions", "transactions", "revenue"]),
        "transactions within sessions": bool((df["transactions"] <= df["sessions"]).all()),
    }
```

```python
clean_checks = validate(clean, START, END)
colleague_checks = validate(colleague, START, END)
assert all(clean_checks.values()), clean_checks
colleague_checks
```

```python
reconciled = clean["sessions"].sum() == REPORTED_SESSIONS
colleague_gap = colleague["sessions"].sum() - REPORTED_SESSIONS
reconciled, colleague_gap
```

</details>